In [21]:
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
import re
from nltk.corpus import stopwords
import nltk
from typing import Dict, List, Tuple
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import torch

# Download NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
stop_words = set(stopwords.words('english'))

STOPWORDS = set(stopwords.words('english'))


In [22]:
# Remove special characters, punctuation, and convert to lowercase
def preprocess_text(text):
    if not isinstance(text, str):
        return ''  # Return empty string for NaN or non-string values
    text = re.sub(r'[^\w\s]', '', text.lower())  # Remove punctuation and convert to lowercase
    text = ' '.join(word for word in text.split() if word not in STOPWORDS)  # Remove stopwords
    return text

In [23]:

# Load the dataset
file_path = "filtered_file_2.xlsx"  # Replace with your actual file path
df = pd.read_excel(file_path)

# Apply text preprocessing
df['Abstract_clean'] = df['Abstract'].apply(preprocess_text)
df['Research_Areas_clean'] = df['Research Areas'].apply(preprocess_text)

# Combine text columns for analysis
df['combined_text'] = df['Abstract_clean'] + ' ' + df['Research_Areas_clean']

# Tokenize the text for FastText training
df['tokenized_text'] = df['combined_text'].apply(lambda x: x.split())



In [24]:
# Print the cleaned text
print(df['Abstract_clean'].head())
print(df['Research_Areas_clean'].head())

0    latent heat storage system two different stack...
1    attempt investigate effect testicular torsion ...
2    longlived fission radionuclides spread chernob...
3    prostheses proposed restore spinal stability p...
4    vertical particle flux measured two sites sout...
Name: Abstract_clean, dtype: object
0                            energy fuels
1                      urology nephrology
2    chemistry nuclear science technology
3                  biophysics engineering
4                            oceanography
Name: Research_Areas_clean, dtype: object


In [25]:
# Function to split multiple researcher IDs into individual rows
def split_researchers(row):
    if not isinstance(row['Researcher Ids'], str):
        return pd.DataFrame()  # Skip invalid entries
    researchers = row['Researcher Ids'].split(';')  # Split by ';'
    data = []
    for researcher_id in researchers:
        data.append({
            'Researcher': researcher_id.strip(),
            'Abstract_clean': row['Abstract_clean'],
            'Research_Areas_clean': row['Research_Areas_clean'],
            'Publication Year': row['Publication Year']  # Keep year for weighting
        })
    return pd.DataFrame(data)

flattened_df = pd.concat([split_researchers(row) for _, row in df.iterrows()], ignore_index=True)

# Combine Abstract and Research Areas for analysis
flattened_df['combined_text'] = flattened_df['Abstract_clean'] + ' ' + flattened_df['Research_Areas_clean']



In [26]:
# Initialize the SentenceTransformer model
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')
# Split researchers and create expanded dataframe
expanded_df = pd.concat([split_researchers(row) for _, row in df.iterrows()], ignore_index=True)


In [27]:
def calculate_cosine_similarity(text1, text2):
    # Convert texts to word frequency vectors
    vectorizer = CountVectorizer().fit_transform([text1, text2])
    vectors = vectorizer.toarray()
    return cosine_similarity(vectors)[0][1]

def calculate_dice_similarity(text1, text2):
    set1 = set(text1.split())
    set2 = set(text2.split())
    intersection = len(set1.intersection(set2))
    return 2 * intersection / (len(set1) + len(set2)) if (len(set1) + len(set2)) > 0 else 0

def calculate_overlap_similarity(text1, text2):
    set1 = set(text1.split())
    set2 = set(text2.split())
    intersection = len(set1.intersection(set2))
    return intersection / min(len(set1), len(set2)) if min(len(set1), len(set2)) > 0 else 0

def calculate_semantic_similarity_batch(texts1, texts2, model, batch_size=32):
    """Calculate semantic similarity in batches"""
    embeddings1 = model.encode(texts1, batch_size=batch_size, convert_to_tensor=True)
    embeddings2 = model.encode(texts2, batch_size=batch_size, convert_to_tensor=True)
    
    # Calculate similarities in batches
    similarities = torch.cosine_similarity(embeddings1.unsqueeze(1), embeddings2.unsqueeze(0), dim=2)
    return similarities.diagonal().cpu().numpy()

In [28]:
def calculate_article_weight(count):
    if count < 10:
        # Low number of articles gets lower weight but still considerable
        return 0.8 + (count * 0.02)  # Starts at 0.8, increases gradually
    elif count <= 20:
        # Sweet spot - highest weights for 15-20 articles
        return 1.0 + ((count - 10) * 0.04)  # Peaks at 1.4 for 20 articles
    elif count <= 35:
        # Gradually decreasing weight but still good
        return 1.4 - ((count - 20) * 0.02)  # Decreases from 1.4 to 1.1
    else:
        # Penalty for high counts, but not too severe
        excess = count - 35
        return 1.1 * np.exp(-0.005 * excess)  # Gentler exponential decay

# Update the find_matching_researchers function with this new weight calculation
def find_matching_researchers(project_idea: str, top_n: int = 5):
    print("Processing project idea...")
    project_idea_clean = preprocess_text(project_idea)

    # Create a working copy of the flattened dataframe
    results_df = flattened_df.copy()
    
    print("Calculating cosine similarities...")
    results_df['similarity'] = results_df['combined_text'].apply(
        lambda x: calculate_cosine_similarity(project_idea_clean, x))

    print("Calculating final scores...")
    article_counts = results_df['Researcher'].value_counts()
    results_df['article_count'] = results_df['Researcher'].map(article_counts)
    
    results_df['article_weight'] = results_df['article_count'].apply(calculate_article_weight)
    
    # Calculate year weights
    current_year = 2025
    results_df['year_weight'] = 1 / (1 + (np.log(1 + current_year - results_df['Publication Year']) / np.log(100)))
    
    # Calculate final score
    results_df['final_score'] = (
        (results_df['similarity'] * 1.5) * 
        results_df['year_weight'] * 
        results_df['article_weight']
    )
    
    # Group and sort
    grouped_results = results_df.groupby('Researcher').agg({
        'final_score': 'max',
        'article_count': 'first',
        'year_weight': 'max',
        'article_weight': 'first',
        'Publication Year': 'max'
    }).reset_index()
    
    return grouped_results.sort_values('final_score', ascending=False).head(top_n)

In [29]:
# Example usage
project_idea = "cell cancer bio bioinformatics machine learning algorithm human life gene gen heredity"

print("Finding matching researchers...")
top_matches = find_matching_researchers(project_idea, top_n=10)  # Removed model=sentence_model

print("\nTop Recommended Collaborators:")
print(top_matches[['Researcher', 'final_score', 'article_count', 'year_weight', 'Publication Year']])

# Search for specific researcher if needed
specific_researcher = top_matches[top_matches['Researcher'].str.contains('ABG-9392-2021', case=False, na=False)]
if not specific_researcher.empty:
    print("\nSpecific Researcher Details:")
    print(specific_researcher[['Researcher', 'final_score', 'article_count', 'year_weight', 'Publication Year']])

Finding matching researchers...
Processing project idea...
Calculating cosine similarities...
Calculating final scores...

Top Recommended Collaborators:
                                  Researcher  final_score  article_count  \
7951              Işık, Zerrin/ABG-9392-2021     0.397660             22   
6463   Georgakilas, Alexandros/AAG-9517-2020     0.395008             20   
1221      Arayici, Mehmet Emin/ABB-2538-2020     0.362572             28   
17407             Yuce, Zeynep/JDW-8153-2023     0.351978             37   
17264           Yildirim, Nuri/JXX-4446-2024     0.350876             15   
10019          Leblebici, Asım/HHM-5895-2022     0.339180             14   
1011              Alotaibi, Hani/R-3158-2016     0.327292             14   
2627       Birant, Kökten Ulaş/AAD-8386-2022     0.327130             16   
14234               Senol, Yavuz/P-9848-2019     0.323817             14   
14626             Soner, Burak/AAB-7965-2020     0.316732              9   

       ye